# Complejidad Parametrizada: k-Vertex Cover

Este cuaderno implementa algoritmos FPT para el problema k-Vertex Cover,
ilustrando el artículo
[Complejidad Parametrizada](../../04-complejidad-computacional/10-complejidad-parametrizada.md).

Técnicas implementadas:
- **Árbol de búsqueda acotado**: O(2^k · n)
- **Kernelización**: reduce la instancia a O(k²) vértices antes de buscar

In [ ]:
import random
import time
from collections import defaultdict

## Representación de grafos

In [ ]:
def make_graph(n_vertices, edges):
    """Crea un grafo como lista de adyacencia (conjunto de vecinos)."""
    adj = defaultdict(set)
    for u, v in edges:
        adj[u].add(v)
        adj[v].add(u)
    # Aseguramos que todos los vértices existen
    for i in range(n_vertices):
        _ = adj[i]  # crea entrada vacía si no existe
    return adj


def degree(adj, v):
    return len(adj[v])


def is_vertex_cover(adj, cover):
    """Verifica si cover es un vertex cover válido."""
    cover_set = set(cover)
    for u in adj:
        for v in adj[u]:
            if u not in cover_set and v not in cover_set:
                return False
    return True


def remove_vertex(adj, v):
    """Retorna copia del grafo sin el vértice v y sus aristas."""
    new_adj = defaultdict(set)
    for u in adj:
        if u != v:
            new_adj[u] = adj[u] - {v}
    return new_adj


def pick_edge(adj):
    """Elige cualquier arista del grafo, o None si no hay aristas."""
    for u in adj:
        for v in adj[u]:
            if u < v:
                return (u, v)
    return None


# Grafo de ejemplo: K4 (clique de 4 vértices)
k4_edges = [(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)]
G_k4 = make_graph(4, k4_edges)
print("K4:", dict(G_k4))
# El vertex cover mínimo de K4 tiene 3 vértices
print("¿{0,1,2} es VC?", is_vertex_cover(G_k4, {0,1,2}))

## Árbol de búsqueda acotado

Dado una arista (u,v), al menos uno de los extremos debe estar en el cover.
El algoritmo hace branching: incluir u ó incluir v, reduciendo k en 1 cada vez.

In [ ]:
def vertex_cover_bst(adj, k):
    """Árbol de búsqueda acotado para k-Vertex Cover.
    
    Retorna un vertex cover de tamaño <= k, o None si no existe.
    Complejidad: O(2^k * n).
    """
    # Caso base 1: no hay aristas → cover vacío es válido
    edge = pick_edge(adj)
    if edge is None:
        return set()
    
    # Caso base 2: presupuesto agotado pero quedan aristas
    if k == 0:
        return None
    
    u, v = edge
    
    # Rama 1: incluir u
    adj_u = remove_vertex(adj, u)
    result_u = vertex_cover_bst(adj_u, k - 1)
    if result_u is not None:
        return result_u | {u}
    
    # Rama 2: incluir v
    adj_v = remove_vertex(adj, v)
    result_v = vertex_cover_bst(adj_v, k - 1)
    if result_v is not None:
        return result_v | {v}
    
    return None


def min_vertex_cover_bst(adj):
    """Encuentra el vertex cover mínimo probando k=0,1,2,..."""
    n = len(adj)
    for k in range(n + 1):
        result = vertex_cover_bst(adj, k)
        if result is not None:
            return result, k
    return set(adj.keys()), n


# Test con K4
cover, k = min_vertex_cover_bst(G_k4)
print(f"K4: vertex cover mínimo = {cover}, tamaño = {k}")
print(f"¿Válido? {is_vertex_cover(G_k4, cover)}")

# Test con camino P5
p5_edges = [(0,1),(1,2),(2,3),(3,4)]
G_p5 = make_graph(5, p5_edges)
cover, k = min_vertex_cover_bst(G_p5)
print(f"\nP5: vertex cover mínimo = {cover}, tamaño = {k}")
print(f"¿Válido? {is_vertex_cover(G_p5, cover)}")

## Kernelización para k-Vertex Cover

Reglas de reducción:
1. Si deg(v) > k, v debe estar en el cover → incluirlo, reducir k.
2. Si quedan más de k² aristas tras aplicar la regla 1 → NO (imposible).
3. El kernel resultante tiene ≤ 2k² vértices y ≤ k² aristas.

In [ ]:
def kernelize_vc(adj, k):
    """Aplica kernelización al problema k-Vertex Cover.
    
    Retorna (kernel_adj, k_nuevo, forced_cover) donde:
    - kernel_adj: grafo reducido
    - k_nuevo: presupuesto restante
    - forced_cover: vértices que deben estar en el cover
    - None si la instancia es NO (imposible)
    """
    adj = defaultdict(set, {v: set(nb) for v, nb in adj.items()})
    forced = set()
    remaining_k = k
    
    changed = True
    while changed:
        changed = False
        # Regla 1: vértice de grado > k debe entrar
        high_deg = [v for v in list(adj.keys()) if len(adj[v]) > remaining_k]
        for v in high_deg:
            if remaining_k <= 0:
                return None  # imposible
            forced.add(v)
            remaining_k -= 1
            adj = remove_vertex(adj, v)
            changed = True
    
    # Regla 2: contar aristas en kernel
    n_edges = sum(len(nb) for nb in adj.values()) // 2
    if n_edges > remaining_k ** 2:
        return None  # imposible: demasiadas aristas
    
    return adj, remaining_k, forced


def vertex_cover_with_kernel(adj, k):
    """k-Vertex Cover con kernelización + árbol de búsqueda acotado."""
    result = kernelize_vc(adj, k)
    if result is None:
        return None
    kernel_adj, k_remaining, forced = result
    
    # Resolver en el kernel
    kernel_cover = vertex_cover_bst(kernel_adj, k_remaining)
    if kernel_cover is None:
        return None
    
    return forced | kernel_cover


# Test: grafo estrella K_{1,5} (el centro tiene grado 5)
star_edges = [(0, i) for i in range(1, 6)]
G_star = make_graph(6, star_edges)

cover = vertex_cover_with_kernel(G_star, 2)
print("K_{1,5} con k=2:")
print(f"  Cover: {cover}")
print(f"  ¿Válido? {is_vertex_cover(G_star, cover) if cover else 'N/A (NO instancia)'}")

# Con k=1 debería encontrar solo el centro
cover1 = vertex_cover_with_kernel(G_star, 1)
print(f"K_{{1,5}} con k=1: cover={cover1}, válido={is_vertex_cover(G_star, cover1) if cover1 else 'NO'}")

## Ejercicio 1: Implementa la búsqueda exhaustiva (fuerza bruta)

Para comparar con BST, implementa el algoritmo de fuerza bruta:
prueba todos los subconjuntos de vértices de tamaño k.

In [ ]:
def subsets_of_size_k(elements, k):
    """Genera todos los subconjuntos de 'elements' de tamaño k."""
    elements = list(elements)
    n = len(elements)
    if k == 0:
        yield []
        return
    if k > n:
        return
    for i in range(n):
        for rest in subsets_of_size_k(elements[i+1:], k-1):
            yield [elements[i]] + rest


def vertex_cover_brute(adj, k):
    """TODO: Implementa la búsqueda exhaustiva.
    
    Para cada subconjunto de vértices de tamaño k, verifica si
    es un vertex cover válido. Retorna el primer cover encontrado
    o None si no existe.
    
    Complejidad: O(C(n,k) * m) donde m es el número de aristas.
    """
    # TODO: Tu implementación aquí
    pass


# --- Solución de referencia (comentada) ---
# def vertex_cover_brute_ref(adj, k):
#     vertices = list(adj.keys())
#     for subset in subsets_of_size_k(vertices, k):
#         if is_vertex_cover(adj, set(subset)):
#             return set(subset)
#     return None

# Test
# cover_bf = vertex_cover_brute(G_k4, 3)
# print(f"Brute force K4 k=3: {cover_bf}")
print("Implementa vertex_cover_brute y descomenta el test.")

## Ejercicio 2: Comparación de tiempos BST vs. fuerza bruta

In [ ]:
def random_graph(n, edge_prob, seed=None):
    """Grafo aleatorio G(n, p) de Erdős-Rényi."""
    if seed is not None:
        random.seed(seed)
    edges = [(i, j) for i in range(n) for j in range(i+1, n)
             if random.random() < edge_prob]
    return make_graph(n, edges)


def benchmark_vc(n, edge_prob, k, seed=42):
    """Compara tiempos de BST vs. fuerza bruta."""
    G = random_graph(n, edge_prob, seed)
    
    t0 = time.perf_counter()
    cover_bst = vertex_cover_with_kernel(G, k)
    t_bst = time.perf_counter() - t0
    
    # Fuerza bruta (solo si n y k son pequeños)
    t_bf = None
    if n <= 20 and k <= 8:
        # Descomenta cuando tengas vertex_cover_brute implementado:
        # t0 = time.perf_counter()
        # cover_bf = vertex_cover_brute(G, k)
        # t_bf = time.perf_counter() - t0
        pass
    
    n_edges = sum(len(nb) for nb in G.values()) // 2
    found = cover_bst is not None
    return n_edges, found, t_bst, t_bf


print("Benchmark k-Vertex Cover con árbol de búsqueda acotado")
print(f"{'n':>4} | {'k':>3} | {'aristas':>7} | {'encontrado':>10} | {'t_BST (ms)':>12}")
print("-" * 50)
random.seed(0)
for n, k in [(15, 5), (20, 7), (25, 8), (30, 10)]:
    n_edges, found, t_bst, _ = benchmark_vc(n, 0.3, k)
    print(f"{n:>4} | {k:>3} | {n_edges:>7} | {str(found):>10} | {t_bst*1000:>12.2f}")

## Ejercicio 3: Tamaño del kernel

In [ ]:
def kernel_size(adj, k):
    """TODO: Retorna (n_vertices_kernel, n_edges_kernel) tras kernelización.
    
    Usa kernelize_vc para obtener el kernel y cuenta sus vértices y aristas.
    Si la instancia es NO, retorna (0, 0).
    """
    # TODO: Tu implementación aquí
    pass


# --- Solución de referencia (comentada) ---
# def kernel_size_ref(adj, k):
#     result = kernelize_vc(adj, k)
#     if result is None:
#         return (0, 0)
#     kernel_adj, k_new, forced = result
#     n_v = sum(1 for v in kernel_adj if len(kernel_adj[v]) > 0 or v in kernel_adj)
#     n_v = len(kernel_adj)
#     n_e = sum(len(nb) for nb in kernel_adj.values()) // 2
#     return (n_v, n_e)

# Test esperado: el kernel tiene <= k^2 aristas
# G_test = random_graph(50, 0.15, seed=1)
# for k in [5, 8, 10]:
#     nv, ne = kernel_size_ref(G_test, k)  # O usa tu implementación
#     print(f"k={k}: kernel tiene {nv} vértices, {ne} aristas (límite: {k**2})")
print("Implementa kernel_size y descomenta el test.")

## Ideas clave

- El **árbol de búsqueda acotado** explota la estructura del problema:
  toda arista obliga a incluir al menos un extremo → branching de factor 2.
- La **kernelización** reduce la instancia a O(k²) antes de buscar:
  vértices de grado alto DEBEN estar en el cover.
- FPT significa que la dependencia exponencial queda confinada al parámetro k;
  para k pequeño el algoritmo es práctico aunque n sea grande.
- La composición kernel + BST da tiempo O(2^k · k²) + preprocesamiento polinomial.

## Soluciones de referencia

In [ ]:
# Ejercicio 1
def vertex_cover_brute_ref(adj, k):
    vertices = list(adj.keys())
    for subset in subsets_of_size_k(vertices, k):
        if is_vertex_cover(adj, set(subset)):
            return set(subset)
    return None

cover_bf = vertex_cover_brute_ref(G_k4, 3)
print(f"Ref Ejercicio 1 — K4 k=3: {cover_bf}, válido: {is_vertex_cover(G_k4, cover_bf)}")

# Ejercicio 3
def kernel_size_ref(adj, k):
    result = kernelize_vc(adj, k)
    if result is None:
        return (0, 0)
    kernel_adj, k_new, forced = result
    n_v = len(kernel_adj)
    n_e = sum(len(nb) for nb in kernel_adj.values()) // 2
    return (n_v, n_e)

print("\nRef Ejercicio 3 — tamaños de kernel:")
G_test = random_graph(50, 0.15, seed=1)
for k in [5, 8, 10]:
    nv, ne = kernel_size_ref(G_test, k)
    print(f"  k={k}: kernel={nv} vértices, {ne} aristas (límite k²={k**2})")